<a href="https://colab.research.google.com/github/vasireddysrimanth/House-Price-Prediction-Using-PyTorch/blob/develop/House_Price_Prediction_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# House Price Prediction Using PyTorch

*   List item
*   List item



**A Complete Beginner-Friendly Practical · Google Colab**

In this practical, we will build, train, evaluate, save, and reload a simple neural network using PyTorch to predict house prices.

The model uses three input features:

- **House Size** — the size of the house.
- **Number of Bedrooms** — the number of bedrooms.
- **House Age** — the age of the house in years.

The target value is the house price, expressed in **lakhs**.

> This is a small, illustrative dataset created for learning purposes. It is not intended to represent real-world property prices.

## Step 1: Import Required Libraries

**What are we doing?**

We begin by importing the PyTorch libraries required to build and train our neural network. We also import `TensorDataset`, `DataLoader`, and `random_split` to organize the data, create batches, and split the dataset into training and testing subsets.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

## Step 2: Create the Dataset (Input Features)

**What are we doing?**

Next, we create the input dataset. Each row represents one house, and each column represents a feature: house size, number of bedrooms, and house age.

The input tensor `X` therefore contains three features for every house.

In [ ]:
X = torch.tensor([
    [1000, 2, 5],
    [1500, 3, 3],
    [800, 2, 10],
    [2000, 4, 2],
    [1200, 2, 7],
    [1800, 3, 4],
    [900, 2, 8],
    [2500, 4, 1]
], dtype=torch.float32)

## Step 3: Create the Target Values

**What are we doing?**

Now, we define the actual house price corresponding to each row in the input dataset.

The target tensor `y` contains one price per house. These values are expressed in lakhs and will be used by the model during training to learn the relationship between house features and prices.

In [ ]:
y = torch.tensor([
    [45], [70], [30], [100],
    [50], [85], [35], [125]
], dtype=torch.float32)

## Step 4: Inspect the Dataset Shapes

**What are we doing?**

Before training, we inspect the dimensions of the input and target tensors.

- `X.shape` shows the number of samples and input features.
- `y.shape` shows the number of samples and target values.
- We also print the first house and its corresponding price to verify that the data is structured correctly.

In [ ]:
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)
print("First House:", X[0])
print("First Price:", y[0])

## Step 5: Combine the Inputs and Targets

**What are we doing?**

We use PyTorch's `TensorDataset` to combine the input features (`X`) and target values (`y`) into a single dataset.

This keeps each house's features paired with its corresponding price. We then print the total number of samples in the dataset.

In [ ]:
dataset = TensorDataset(X, y)
print("Total Samples:", len(dataset))

## Step 6: Split the Dataset into Training and Testing Sets

**What are we doing?**

We divide the dataset into two subsets:

- **Training set (80%)** — used to train the neural network.
- **Testing set (20%)** — used to evaluate the model on samples that were not used for training.

We set a random seed with `manual_seed(42)` so that the split is reproducible when the notebook is run again.

In [ ]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("Training Data:", len(train_dataset))
print("Testing Data:", len(test_dataset))

## Step 7: Separate Training and Testing Features

**What are we doing?**

The `random_split` function returns subsets containing the original dataset indices. We use those indices to retrieve the corresponding input features and target values.

This gives us four separate tensors: `X_train`, `X_test`, `y_train`, and `y_test`.

In [ ]:
X_train = X[train_dataset.indices]
X_test = X[test_dataset.indices]

y_train = y[train_dataset.indices]
y_test = y[test_dataset.indices]

## Step 8: Normalize the Input Features

**What are we doing?**

The input features have different numerical ranges. For example, house size may be in the thousands, while the number of bedrooms is a much smaller number.

Normalization puts the features on a comparable scale. We calculate the mean and standard deviation using only the training data, then apply those same values to both the training and testing features. This prevents information from the test set from influencing the normalization process.

In [ ]:
mean = X_train.mean(dim=0)
std = X_train.std(
    dim=0,
    unbiased=False
).clamp_min(1e-8)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print("Mean:", mean)
print("Standard Deviation:", std)
print("Normalized Training Data:", X_train)

## Step 9: Create DataLoaders

**What are we doing?**

We create a `TensorDataset` for each normalized subset and use PyTorch's `DataLoader` to provide the data in batches.

With `batch_size=2`, each training iteration processes two samples at a time.

- `shuffle=True` randomizes the training samples at the beginning of each epoch.
- `shuffle=False` keeps the testing samples in a consistent order.

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

print("Training Batches:", len(train_loader))
print("Testing Batches:", len(test_loader))

## Step 10: Build the Neural Network Model

**What are we doing?**

We define a neural network by creating a class that inherits from `nn.Module`.

The network receives three input features and passes them through two hidden layers before producing one output: the predicted house price.

**Network architecture:** `3 → 16 → 8 → 1`

- Input layer: 3 features.
- First hidden layer: 16 neurons.
- Second hidden layer: 8 neurons.
- Output layer: 1 predicted price.
- ReLU: introduces non-linearity between the hidden layers.

In [ ]:
class HousePriceModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(3, 16)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.output(x)
        return x

## Step 11: Initialize the Model

**What are we doing?**

We create an instance of the `HousePriceModel` class. When the model is initialized, PyTorch creates the layer weights and biases.

Printing the model displays its layer structure so we can verify the architecture.

In [ ]:
model = HousePriceModel()
print(model)

## Step 12: Define the Loss Function

**What are we doing?**

We use Mean Squared Error (MSE) as the loss function.

MSE calculates the average squared difference between the model's predicted prices and the actual prices. During training, the model attempts to reduce this error.

In [ ]:
loss_fn = nn.MSELoss()

## Step 13: Define the Optimizer

**What are we doing?**

We use the Adam optimizer to update the model's weights and biases during training.

The learning rate (`lr=0.001`) controls the size of the updates made by the optimizer. The optimizer uses the model's trainable parameters to reduce the loss over time.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## Step 14: Train the Model

**What are we doing?**

We train the neural network for 500 epochs. One epoch means the model has processed the entire training dataset once.

For each batch, the training loop performs these operations:

1. Clear gradients from the previous iteration using `optimizer.zero_grad()`.
2. Generate predictions using the model.
3. Calculate the loss by comparing predictions with actual prices.
4. Calculate gradients using `loss.backward()`.
5. Update the model parameters using `optimizer.step()`.

We print the average training loss every 50 epochs to monitor the learning process.

In [ ]:
epochs = 500

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()

        prediction = model(batch_X)
        loss = loss_fn(prediction, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Loss: {avg_loss:.4f}"
        )

## Step 15: Evaluate the Model

**What are we doing?**

After training, we evaluate the model using the testing dataset.

We call `model.eval()` to switch the model to evaluation mode and use `torch.no_grad()` to disable gradient calculations. This evaluation loop calculates the average test loss without updating the model's weights.

In [ ]:
model.eval()
test_loss = 0.0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        prediction = model(batch_X)
        loss = loss_fn(prediction, batch_y)
        test_loss += loss.item()

avg_test_loss = test_loss / len(test_loader)
print("Test Loss:", avg_test_loss)

## Step 16: Make a Prediction for a New House

**What are we doing?**

We now test the trained model with a new house that has a size of 1,500, three bedrooms, and an age of five years.

It is important to normalize this new input using the training set's original mean and standard deviation. We then pass the normalized input to the model and display the predicted price in lakhs.

In [ ]:
new_house = torch.tensor(
    [[1500, 3, 5]],
    dtype=torch.float32
)

# Apply the same normalization used for the training data
new_house = (new_house - mean) / std

model.eval()
with torch.no_grad():
    predicted_price = model(new_house)

print(
    "Predicted House Price:",
    predicted_price.item(),
    "Lakhs"
)

## Step 17: Save the Trained Model

**What are we doing?**

Once training is complete, we save the model's learned parameters using `state_dict()`.

The parameters are stored in a file named `house_price_model.pth`. Saving the state dictionary allows us to restore the learned weights later without training the model again.

In [ ]:
torch.save(
    model.state_dict(),
    "house_price_model.pth"
)

print("Model Saved Successfully!")

## Step 18: Save the Normalization Values

**What are we doing?**

The model requires the same normalization values whenever it makes predictions on new data.

Therefore, we save the training mean and standard deviation in a separate file named `normalization.pth`. These values must be reused when preparing future inputs.

In [ ]:
torch.save(
    {
        "mean": mean,
        "std": std
    },
    "normalization.pth"
)

## Step 19: Load the Trained Model

**What are we doing?**

We create a new model instance and restore its learned parameters from `house_price_model.pth`.

We also load the saved normalization values from `normalization.pth`. The loaded mean and standard deviation will be used to preprocess new house data in the next step.

`map_location="cpu"` ensures that the saved tensors are loaded onto the CPU. `weights_only=True` loads the saved tensor weights safely with the supported PyTorch versions.

In [ ]:
loaded_model = HousePriceModel()

loaded_model.load_state_dict(
    torch.load(
        "house_price_model.pth",
        map_location="cpu",
        weights_only=True
    )
)

normalization = torch.load(
    "normalization.pth",
    map_location="cpu",
    weights_only=True
)

loaded_mean = normalization["mean"]
loaded_std = normalization["std"]

loaded_model.eval()
print("Model Loaded Successfully!")

## Step 20: Make a Prediction Using the Loaded Model

**What are we doing?**

Finally, we use the reloaded model to predict the price of a new house.

We normalize the input using the saved training mean and standard deviation, then pass it to the loaded model. This confirms that the saved model and preprocessing values can be reused to make predictions without repeating the training process.

In [ ]:
new_house = torch.tensor(
    [[1500, 3, 5]],
    dtype=torch.float32
)

new_house = (new_house - loaded_mean) / loaded_std

with torch.no_grad():
    prediction = loaded_model(new_house)

print("Predicted Price:", prediction.item(), "Lakhs")

## Practical Complete

You have completed all 20 steps of a beginner-friendly house price prediction project using PyTorch.

You have learned how to:

- Prepare input features and target values.
- Split data into training and testing sets.
- Normalize input features.
- Build a neural network using `nn.Module`.
- Train the model using a loss function and optimizer.
- Evaluate the model on testing data.
- Make predictions for new house details.
- Save and reload model parameters and normalization values.

**Note:** This notebook uses a very small illustrative dataset to demonstrate the PyTorch workflow. A real-world house price prediction system would require a larger, representative dataset and more thorough evaluation.